# Traffic Event Retrieval for Route Optimization and Traffic Intelligence

## Data Ingestion and Corpus Building 

### Traffic Weather Temporal Data Loader

In [62]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional,Any
import logging
import os
import pickle
import networkx as nx
from collections import Counter, defaultdict
from datetime import datetime
import math
import json
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download required NLTK data (only needed once)
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')



In [63]:
class TrafficWeatherDataLoader:
    """Load and analyze traffic-weather temporal data"""
    
    def __init__(self, data_path: str):
        self.data_path = Path(data_path)
        self.traffic_data = None
        self.data_stats = {}
        
    @staticmethod    
    def get_data_path(filename):
        """Get data path with multiple fallback options"""
        possible_paths = [
            f"./data/{filename}",
            f"../data/{filename}",
            f"../../data/{filename}",
            f"/kaggle/input/datasets/lh7ng0cifjog1/routeiq/{filename}",
            f"/kaggle/input/datasets/lh7ng0cifjog1/routiq/{filename}",
            filename,
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                print(f"Found data at: {path}")
                return path
        
        print(f"Data file not found: {filename}")
        print("Please ensure the data file is available")
        return filename
        
    def load_traffic_weather_data(self, filename: str = "traffic_weather_temporal.csv") -> pd.DataFrame:
        """Load the main traffic-weather dataset"""
        print("Loading Traffic-Weather Temporal Data...")
        
        # Use the path resolution method
        file_path = self.get_data_path(filename)
        file_path = Path(file_path)  # Convert to Path object
        
        try:
            # Load with optimized parameters for large dataset
            self.traffic_data = pd.read_csv(
                file_path,
                parse_dates=['timestamp', 'datetime'],
                dtype={
                    'source_node': 'int32',
                    'target_node': 'int32',
                    'day_of_week': 'int8',
                    'hour_of_day': 'int8',
                    'weather_code': 'int8',
                    'is_rain': 'int8',
                    'is_heavy_rain': 'int8',
                    'is_weekend': 'bool',
                    'is_rush_hour': 'bool'
                }
            )
            
            print(f"Loaded {len(self.traffic_data):,} traffic records")
            print(f"Time range: {self.traffic_data['timestamp'].min()} to {self.traffic_data['timestamp'].max()}")
            print(f"Network: {self.traffic_data['source_node'].nunique()} source nodes, {self.traffic_data['target_node'].nunique()} target nodes")
            
            return self.traffic_data
            
        except FileNotFoundError:
            print(f"Error: File not found at {file_path}")
            print(f"Attempted path: {file_path}")
            raise
        except Exception as e:
            print(f"Error loading data: {e}")
            raise
    
    def analyze_data_quality(self) -> Dict:
        """Analyze data quality and generate statistics"""
        if self.traffic_data is None:
            raise ValueError("Data not loaded. Call load_traffic_weather_data() first.")
        
        print("\nData Quality Analysis...")
        
        stats = {
            'total_rows': len(self.traffic_data),
            'total_columns': len(self.traffic_data.columns),
            'memory_usage_mb': self.traffic_data.memory_usage(deep=True).sum() / 1024**2,
            'missing_values': self.traffic_data.isnull().sum().sum(),
            'duplicate_rows': self.traffic_data.duplicated().sum(),
            'date_range': {
                'start': self.traffic_data['timestamp'].min(),
                'end': self.traffic_data['timestamp'].max(),
                'hours': (self.traffic_data['timestamp'].max() - self.traffic_data['timestamp'].min()).total_seconds() / 3600
            },
            'network_stats': {
                'unique_sources': self.traffic_data['source_node'].nunique(),
                'unique_targets': self.traffic_data['target_node'].nunique(),
                'unique_pairs': self.traffic_data[['source_node', 'target_node']].drop_duplicates().shape[0]
            },
            'weather_stats': {
                'rainy_records': self.traffic_data[self.traffic_data['is_rain'] == 1].shape[0],
                'heavy_rain_records': self.traffic_data[self.traffic_data['is_heavy_rain'] == 1].shape[0],
                'avg_temperature': self.traffic_data['temperature'].mean(),
                'avg_precipitation': self.traffic_data['precipitation'].mean()
            },
            'traffic_stats': {
                'rush_hour_records': self.traffic_data[self.traffic_data['is_rush_hour'] == True].shape[0],
                'weekend_records': self.traffic_data[self.traffic_data['is_weekend'] == True].shape[0],
                'avg_vehicle_count': self.traffic_data['vehicle_counts'].mean(),
                'max_vehicle_count': self.traffic_data['vehicle_counts'].max()
            }
        }
        
        self.data_stats = stats
        
        # Print summary
        print(f"Dataset Size: {stats['total_rows']:,} rows × {stats['total_columns']} columns")
        print(f"Memory Usage: {stats['memory_usage_mb']:.1f} MB")
        print(f"Missing Values: {stats['missing_values']}")
        print(f"Duplicate Rows: {stats['duplicate_rows']}")
        print(f"Time Coverage: {stats['date_range']['hours']:.1f} hours")
        print(f"Rainy Records: {stats['weather_stats']['rainy_records']:,} ({stats['weather_stats']['rainy_records']/stats['total_rows']*100:.1f}%)")
        print(f"Rush Hour Records: {stats['traffic_stats']['rush_hour_records']:,}")
        
        return stats
    
    def get_sample_data(self, n_samples: int = 1000, random_state: int = 42) -> pd.DataFrame:
        """Get a sample of the data for development"""
        if self.traffic_data is None:
            raise ValueError("Data not loaded. Call load_traffic_weather_data() first.")
        
        if len(self.traffic_data) <= n_samples:
            return self.traffic_data.copy()
        
        return self.traffic_data.sample(n=n_samples, random_state=random_state)

### Network Data Loader

In [64]:
class KigaliNetworkLoader:
    """Load and analyze Kigali road network data"""
    
    def __init__(self, data_path: str):
        self.data_path = Path(data_path)
        self.network_graph = None
        self.network_stats = {}

    @staticmethod    
    def get_data_path(filename: str) -> str:
        """Get data path with multiple fallback options"""
        possible_paths = [
            f"./data/{filename}",
            f"../data/{filename}",
            f"../../data/{filename}",
            f"/kaggle/input/datasets/lh7ng0cifjog1/routeiq/{filename}",  
            f"/kaggle/input/datasets/lh7ng0cifjog1/routiq/{filename}", 
            filename,
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                print(f"Found data at: {path}")
                return path
        
        print(f"Data file not found: {filename}")
        print("Please ensure the data file is available")
        return filename
    
    def load_network_graph(self, filename: str = 'kigali_congested_network.pkl') -> nx.MultiDiGraph:
        """Load the Kigali congested network graph"""
        print("Loading Kigali Road Network Graph...")
        
        file_path = self.get_data_path(filename)
        file_path = Path(file_path)
        
        try:
            with open(file_path, 'rb') as f:
                self.network_graph = pickle.load(f)
            
            print(f"Loaded road network with {self.network_graph.number_of_nodes():,} nodes and {self.network_graph.number_of_edges():,} edges")
            print(f"Graph Type: {type(self.network_graph).__name__}")
            print(f"Connected Components: {nx.number_weakly_connected_components(self.network_graph)}")
            
            return self.network_graph
            
        except FileNotFoundError:
            print(f"Error: Network file not found at {file_path}")
            raise
        except Exception as e:
            print(f"Error loading network: {e}")
            raise
    
    def analyze_network_structure(self) -> Dict[str, Any]:
        """Analyze network topology and structure"""
        if self.network_graph is None:
            raise ValueError("Network not loaded. Call load_network_graph() first.")
        
        print("\nNetwork Structure Analysis...")
        
        G = self.network_graph
        
        stats = {
            'basic_info': {
                'nodes': G.number_of_nodes(),
                'edges': G.number_of_edges(),
                'is_directed': G.is_directed(),
                'is_multigraph': G.is_multigraph(),
                'density': nx.density(G)
            },
            'connectivity': {
                'weakly_connected_components': nx.number_weakly_connected_components(G),
                'strongly_connected_components': nx.number_strongly_connected_components(G),
                'is_weakly_connected': nx.is_weakly_connected(G),
                'is_strongly_connected': nx.is_strongly_connected(G)
            },
            'degree_stats': {
                'avg_degree': sum(dict(G.degree()).values()) / G.number_of_nodes(),
                'max_degree': max(dict(G.degree()).values()),
                'min_degree': min(dict(G.degree()).values()),
                'avg_in_degree': sum(dict(G.in_degree()).values()) / G.number_of_nodes(),
                'avg_out_degree': sum(dict(G.out_degree()).values()) / G.number_of_nodes()
            },
            'node_attributes': self._analyze_node_attributes(),
            'edge_attributes': self._analyze_edge_attributes()  
        }
        
        self.network_stats = stats
        
        print(f"Network: {stats['basic_info']['nodes']:,} nodes, {stats['basic_info']['edges']:,} edges")
        print(f"Density: {stats['basic_info']['density']:.4f}")
        print(f"Avg Degree: {stats['degree_stats']['avg_degree']:.2f}")
        print(f"Connected Components: {stats['connectivity']['weakly_connected_components']}")
        
        return stats
    
    def _analyze_node_attributes(self) -> Dict[str, Any]:
        """Analyze node attributes"""
        G = self.network_graph
        
        if not G.nodes():
            return {}
        
        sample_node = list(G.nodes())[0]
        node_attrs = list(G.nodes[sample_node].keys())
        
        attr_analysis = {'available_attributes': node_attrs}
        
        for attr in node_attrs:
            values = [G.nodes[node].get(attr) for node in G.nodes() if attr in G.nodes[node]]
            
            if values and isinstance(values[0], (int, float)):
                attr_analysis[attr] = {
                    'type': 'numeric',
                    'count': len(values),
                    'mean': np.mean(values),
                    'min': np.min(values),
                    'max': np.max(values)
                }
            else:
                attr_analysis[attr] = {
                    'type': 'categorical',
                    'count': len(values),
                    'unique': len(set(values))
                }
        
        return attr_analysis
    
    def _analyze_edge_attributes(self) -> Dict[str, Any]:
        """Analyze edge attributes with comprehensive coverage"""
        G = self.network_graph
        
        if not G.edges():
            return {}
        
        # Collect ALL attributes from ALL edges
        all_attributes = set()
        edge_data_list = []
        
        for u, v, data in G.edges(data=True):
            edge_data_list.append(data)
            all_attributes.update(data.keys())
        
        if not all_attributes:
            return {'available_attributes': []}
        
        attr_analysis = {'available_attributes': sorted(list(all_attributes))}
        
        print(f"Analyzing {len(edge_data_list)} edges with {len(all_attributes)} attributes")
        
        # Analyze each attribute
        for attr in all_attributes:
            values = []
            edge_count_with_attr = 0
            
            for data in edge_data_list:
                if attr in data:
                    values.append(data[attr])
                    edge_count_with_attr += 1
            
            if not values:
                continue
            
            # Analyze data type distribution
            type_counts = {}
            for val in values:
                val_type = type(val).__name__
                type_counts[val_type] = type_counts.get(val_type, 0) + 1
            
            primary_type = max(type_counts, key=type_counts.get)
            
            if primary_type in ['int', 'float'] and type_counts.get('bool', 0) == 0:
                # Numeric analysis
                numeric_vals = [v for v in values if isinstance(v, (int, float)) and not isinstance(v, bool)]
                
                if numeric_vals:
                    try:
                        attr_analysis[attr] = {
                            'type': 'numeric',
                            'count': len(values),
                            'edge_coverage': edge_count_with_attr / len(edge_data_list) * 100,
                            'numeric_count': len(numeric_vals),
                            'mean': float(np.mean(numeric_vals)),
                            'min': float(np.min(numeric_vals)),
                            'max': float(np.max(numeric_vals)),
                            'std': float(np.std(numeric_vals))
                        }
                    except Exception:
                        attr_analysis[attr] = {
                            'type': 'numeric',
                            'count': len(values),
                            'edge_coverage': edge_count_with_attr / len(edge_data_list) * 100,
                            'numeric_count': len(numeric_vals),
                            'error': 'Could not calculate statistics',
                            'sample_values': numeric_vals[:3]
                        }
                else:
                    attr_analysis[attr] = {
                        'type': 'numeric',
                        'count': len(values),
                        'edge_coverage': edge_count_with_attr / len(edge_data_list) * 100,
                        'numeric_count': 0,
                        'error': 'No valid numeric values found'
                    }
            elif primary_type == 'LineString':
                # Geometry analysis
                geometries = [v for v in values if hasattr(v, 'coords')]
                if geometries:
                    coord_counts = [len(geom.coords) for geom in geometries]
                    attr_analysis[attr] = {
                        'type': 'geometry',
                        'count': len(values),
                        'edge_coverage': edge_count_with_attr / len(edge_data_list) * 100,
                        'geometry_count': len(geometries),
                        'avg_points': float(np.mean(coord_counts)),
                        'min_points': int(np.min(coord_counts)),
                        'max_points': int(np.max(coord_counts))
                    }
                else:
                    attr_analysis[attr] = {
                        'type': 'geometry',
                        'count': len(values),
                        'edge_coverage': edge_count_with_attr / len(edge_data_list) * 100,
                        'geometry_count': 0,
                        'error': 'No valid geometries found'
                    }
            else:
                # Categorical/mixed analysis
                unique_vals = list(set(str(v) for v in values))
                
                attr_analysis[attr] = {
                    'type': 'categorical',
                    'count': len(values),
                    'edge_coverage': edge_count_with_attr / len(edge_data_list) * 100,
                    'unique': len(unique_vals),
                    'type_distribution': type_counts,
                    'sample_values': unique_vals[:10]
                }
        
        return attr_analysis
        
    def get_node_coordinates(self) -> Dict[int, Tuple[float, float]]:
        """Extract node coordinates for spatial analysis"""
        if self.network_graph is None:
            raise ValueError("Network not loaded.")
        
        coordinates = {}
        for node in self.network_graph.nodes():
            node_data = self.network_graph.nodes[node]
            if 'x' in node_data and 'y' in node_data:
                coordinates[node] = (node_data['x'], node_data['y'])
        
        print(f"Extracted coordinates for {len(coordinates)} nodes")
        return coordinates
    
    def inspect_all_edge_attributes(self):
        """Inspect all edge attributes in detail"""
        G = self.network_graph
        
        if not G.edges():
            print("No edges found")
            return
        
        print(f"Inspecting {G.number_of_edges()} edges...")
        
        all_attrs = set()
        attr_examples = {}
        attr_coverage = {}
        
        for i, (u, v, data) in enumerate(G.edges(data=True)):
            for attr, value in data.items():
                if attr not in all_attrs:
                    all_attrs.add(attr)
                    attr_examples[attr] = value
                    attr_coverage[attr] = 1
                else:
                    attr_coverage[attr] += 1
            
            if i < 3:
                print(f"\nEdge {i+1}: {u} -> {v}")
                print(f"  Attributes: {list(data.keys())}")
                for attr, val in data.items():
                    if isinstance(val, str) and len(val) > 50:
                        print(f"    {attr}: {type(val).__name__} (length: {len(val)})")
                    else:
                        print(f"    {attr}: {val} ({type(val).__name__})")
        
        print(f"\nSummary of all {len(all_attrs)} attributes:")
        for attr in sorted(all_attrs):
            coverage_pct = (attr_coverage[attr] / G.number_of_edges()) * 100
            example = attr_examples[attr]
            example_str = str(example)[:50] + "..." if len(str(example)) > 50 else str(example)
            print(f"  {attr}: {attr_coverage[attr]} edges ({coverage_pct:.1f}%) - Example: {example_str}")


## Document Transformation Engine

### Traffic Event Document Generator

In [65]:
class TrafficEventDocumentGenerator:
    """Transform traffic data rows into searchable IR documents"""
    
    def __init__(self, network_loader: Optional[KigaliNetworkLoader] = None):
        self.network_loader = network_loader
        # Only build network lookup if provided and needed
        if network_loader and network_loader.network_graph:
            self.edge_attributes = self._build_edge_attribute_lookup()
        else:
            self.edge_attributes = {}
    
    def _build_edge_attribute_lookup(self) -> Dict:
        """Build lookup table for edge attributes"""
        if not self.network_loader or not self.network_loader.network_graph:
            return {}
        
        edge_lookup = {}
        G = self.network_loader.network_graph
        
        for u, v, data in G.edges(data=True):
            edge_key = f"{u}_{v}"
            edge_lookup[edge_key] = {
                'highway': data.get('highway', 'unknown'),
                'name': data.get('name', ''),
                'oneway': data.get('oneway', False),
                'length': data.get('length', 0),
                'lanes': data.get('lanes', ''),
                'maxspeed': data.get('maxspeed', ''),
                'junction': data.get('junction', ''),
                'service': data.get('service', ''),
                'ref': data.get('ref', ''),
                'geometry': data.get('geometry', None)
            }
        
        print(f"Built network edge attribute lookup for {len(edge_lookup)} edges (optional enhancement)")
        return edge_lookup
    
    def _get__attributes(self, source_node: int, target_node: int) -> Dict:
        """Get network attributes if available, otherwise return empty"""
        if not self.edge_attributes:
            return {}
        
        edge_key = f"{source_node}_{target_node}"
        return self.edge_attributes.get(edge_key, {})
    
    def _determine_congestion_level(self, vehicle_count: float, road_capacity: int) -> str:
        """Determine congestion level based on vehicle count and capacity"""
        if road_capacity > 0:
            utilization = vehicle_count / road_capacity
        else:
            utilization = 0
        
        if utilization > 0.8 or vehicle_count > 500:
            return "Heavy Congestion"
        elif utilization > 0.5 or vehicle_count > 200:
            return "Moderate Congestion"
        elif utilization > 0.2 or vehicle_count > 50:
            return "Light Traffic"
        else:
            return "Free Flow"
    
    def _classify_weather_condition(self, row: pd.Series) -> str:
        """Classify weather condition using traffic data features"""
        is_rain = row.get('is_rain', 0)
        is_heavy_rain = row.get('is_heavy_rain', 0)
        precipitation = row.get('precipitation', 0)
        temperature = row.get('temperature', 20)
        visibility = row.get('visibility', 10)
        is_hot = row.get('is_hot', 0)
        is_cold = row.get('is_cold', 0)
        
        if is_heavy_rain == 1 or precipitation > 5.0:
            return "Heavy Rain"
        elif is_rain == 1 or precipitation > 0.1:
            return "Rain"
        elif is_cold == 1 or temperature < 5:
            return "Cold"
        elif is_hot == 1 or temperature > 30:
            return "Hot"
        elif visibility < 5:
            return "Poor Visibility"
        else:
            return "Clear"
    
    def _get_time_context(self, row: pd.Series) -> str:
        """Get time-based context using traffic data features"""
        hour_of_day = int(row['hour_of_day'])
        day_of_week = int(row['day_of_week'])
        is_rush_hour = bool(row['is_rush_hour'])
        is_weekend = bool(row['is_weekend'])
        
        time_contexts = []
        
        if 6 <= hour_of_day <= 9:
            time_contexts.append("Morning Rush")
        elif 17 <= hour_of_day <= 19:
            time_contexts.append("Evening Rush")
        elif 22 <= hour_of_day <= 24 or 0 <= hour_of_day <= 5:
            time_contexts.append("Night")
        elif 10 <= hour_of_day <= 16:
            time_contexts.append("Daytime")
        
        if is_weekend:
            time_contexts.append("Weekend")
        elif day_of_week >= 5:
            time_contexts.append("Friday")
        else:
            time_contexts.append("Weekday")
        
        if is_rush_hour:
            time_contexts.append("Rush Hour")
        
        return " ".join(time_contexts)
    
    def _calculate_distance_from_coordinates(self, row: pd.Series) -> float:
        """Calculate distance using traffic data coordinates"""
        source_x = float(row['source_x'])
        source_y = float(row['source_y'])
        target_x = float(row['target_x'])
        target_y = float(row['target_y'])
        
        # Calculate Euclidean distance and convert to approximate km
        distance = math.sqrt((target_x - source_x)**2 + (target_y - source_y)**2)
        return distance * 111  # Rough conversion to kilometers
    
    def _get_weather_impact_level(self, row: pd.Series) -> str:
        """Determine weather impact level using traffic data features"""
        is_rain = row.get('is_rain', 0)
        is_heavy_rain = row.get('is_heavy_rain', 0)
        precipitation = row.get('precipitation', 0)
        visibility = row.get('visibility', 10)
        wind_speed = row.get('wind_speed', 0)
        
        impact_factors = []
        
        if is_heavy_rain == 1:
            impact_factors.append("Severe Rain Impact")
        elif is_rain == 1 and precipitation > 2.0:
            impact_factors.append("Moderate Rain Impact")
        elif is_rain == 1:
            impact_factors.append("Light Rain Impact")
        
        if visibility < 1:
            impact_factors.append("Very Poor Visibility")
        elif visibility < 5:
            impact_factors.append("Poor Visibility")
        
        if wind_speed > 20:
            impact_factors.append("High Wind Conditions")
        
        if not impact_factors:
            return "Normal Weather Conditions"
        
        return ", ".join(impact_factors)
    
    def create_event_document(self, row: pd.Series, doc_id: str) -> Dict[str, Any]:
        """Transform a single traffic row into a searchable document"""
        
        # Extract key information
        source_node = int(row['source_node'])
        target_node = int(row['target_node'])
        timestamp = row['timestamp']
        vehicle_count = float(row['vehicle_counts'])
        highway_type = str(row['highway_type'])
        road_capacity = int(row['road_capacity'])
        road_length = float(row['road_length_meters'])
        speed_limit = row.get('speed_limit_kmh', 'unknown')
        lanes = row.get('lanes', 'unknown')
        
        # Weather features
        temperature = float(row['temperature'])
        precipitation = float(row['precipitation'])
        humidity = float(row['humidity'])
        pressure = float(row['pressure'])
        wind_speed = float(row['wind_speed'])
        wind_direction = float(row['wind_direction'])
        visibility = float(row['visibility'])
        cloud_cover = float(row['cloud_cover'])
        
        # Time features
        hour_of_day = int(row['hour_of_day'])
        day_of_week = int(row['day_of_week'])
        is_rush_hour = bool(row['is_rush_hour'])
        is_weekend = bool(row['is_weekend'])
        
        # Boolean weather flags
        is_rain = bool(row['is_rain'])
        is_heavy_rain = bool(row['is_heavy_rain'])
        is_hot = bool(row['is_hot'])
        is_cold = bool(row['is_cold'])
        
        # Coordinate features
        source_x = float(row['source_x'])
        source_y = float(row['source_y'])
        target_x = float(row['target_x'])
        target_y = float(row['target_y'])
        
        #  features
        rain_rush_hour = bool(row['rain_rush_hour'])
        rain_weekend = bool(row['rain_weekend'])
        temperature_lag_1h = float(row['temperature_lag_1h'])
        precipitation_lag_1h = float(row['precipitation_lag_1h'])
        
        # Get optional network attributes
        network_attrs = self._get__attributes(source_node, target_node)
        
        road_name = network_attrs.get('name', '')  # Only from network
        network_highway = network_attrs.get('highway', highway_type)  # with network if available
        maxspeed = network_attrs.get('maxspeed', speed_limit)  # with network if available
        junction = network_attrs.get('junction', '')  # Only from network
        service = network_attrs.get('service', '')  # Only from network
        ref = network_attrs.get('ref', '')  # Only from network
        
        # Determine event characteristics
        congestion_level = self._determine_congestion_level(vehicle_count, road_capacity)
        weather_condition = self._classify_weather_condition(row)
        time_context = self._get_time_context(row)
        distance_km = self._calculate_distance_from_coordinates(row)
        weather_impact = self._get_weather_impact_level(row)
        
        # Create comprehensive searchable event text
        event_text_parts = [
            f"{congestion_level} traffic event on {network_highway} road",
            f"from node {source_node} to node {target_node}"
        ]
        
        # Add road name if available from network
        if road_name:
            event_text_parts.append(f"along {road_name}")
        
        # Add road characteristics from traffic data
        road_features = []
        if lanes != 'unknown' and lanes:
            road_features.append(f"{lanes} lanes")
        if maxspeed != 'unknown' and str(maxspeed) != 'nan':
            road_features.append(f"speed limit {maxspeed}")
        if junction:
            road_features.append(f"near {junction}")
        if service:
            road_features.append(f"{service} road")
        if ref:
            road_features.append(f"route {ref}")
        
        if road_features:
            event_text_parts.append(f"({', '.join(road_features)})")
        
        # Add comprehensive weather information
        event_text_parts.append(
            f"Weather: {weather_condition} with {precipitation:.2f}mm precipitation"
        )
        event_text_parts.append(f"Temperature: {temperature:.1f} degrees (humidity: {humidity:.0f}%)")
        event_text_parts.append(f"Wind: {wind_speed:.1f} km/h from {wind_direction:.0f} degrees")
        event_text_parts.append(f"Visibility: {visibility:.1f} km, Pressure: {pressure:.0f} hPa")
        
        # Add traffic information
        event_text_parts.append(
            f"Traffic: {vehicle_count:.0f} vehicles on road with capacity {road_capacity}"
        )
        
        # Add spatial information
        if road_length > 0:
            event_text_parts.append(f"Road length: {road_length:.0f} meters")
        if distance_km > 0:
            event_text_parts.append(f"Distance: {distance_km:.2f} km")
        
        # Add temporal context
        event_text_parts.append(f"Time: {timestamp}. {time_context}")
        
        # Add impact indicators
        impact_indicators = []
        if congestion_level == "Heavy Congestion":
            impact_indicators.append("Severe delays expected")
        if weather_condition in ["Heavy Rain", "Poor Visibility"]:
            impact_indicators.append("Hazardous driving conditions")
        if is_rush_hour:
            impact_indicators.append("Peak traffic congestion")
        if rain_rush_hour:
            impact_indicators.append("Rain during rush hour")
        if rain_weekend:
            impact_indicators.append("Weekend rain conditions")
        if junction == "roundabout":
            impact_indicators.append("Roundabout congestion")
        if service in ["alley", "driveway"]:
            impact_indicators.append("Limited access road")
        
        # Combine all parts
        event_text = ". ".join(event_text_parts) + "."
        
        if impact_indicators:
            event_text += f" Impact: {' '.join(impact_indicators)}."
        
        # Create comprehensive document metadata
        document = {
            'doc_id': doc_id,
            'title': f"{congestion_level} - {weather_condition} - {network_highway}",
            'text': event_text,
            'timestamp': timestamp,
            'datetime': row['datetime'],
            
            # Node and spatial information
            'source_node': source_node,
            'target_node': target_node,
            'source_x': source_x,
            'source_y': source_y,
            'target_x': target_x,
            'target_y': target_y,
            'distance_km': distance_km,
            
            # Traffic characteristics
            'vehicle_count': vehicle_count,
            'congestion_level': congestion_level,
            'highway_type': network_highway,
            'road_name': road_name,
            'road_capacity': road_capacity,
            'road_length_meters': road_length,
            'speed_limit': speed_limit,
            'lanes': lanes,
            
            # Weather characteristics
            'weather_condition': weather_condition,
            'weather_impact': weather_impact,
            'temperature': temperature,
            'temperature_lag_1h': temperature_lag_1h,
            'precipitation': precipitation,
            'precipitation_lag_1h': precipitation_lag_1h,
            'humidity': humidity,
            'pressure': pressure,
            'wind_speed': wind_speed,
            'wind_direction': wind_direction,
            'visibility': visibility,
            'cloud_cover': cloud_cover,
            
            # Weather flags
            'is_rain': is_rain,
            'is_heavy_rain': is_heavy_rain,
            'is_hot': is_hot,
            'is_cold': is_cold,
            
            # Temporal characteristics
            'hour_of_day': hour_of_day,
            'day_of_week': day_of_week,
            'time_context': time_context,
            'is_rush_hour': is_rush_hour,
            'is_weekend': is_weekend,
            'rain_rush_hour': rain_rush_hour,
            'rain_weekend': rain_weekend,
            
            # Other features
            'day_of_week_num': int(row['day_of_week_num']),
            'hour_sin': float(row['hour_sin']),
            'hour_cos': float(row['hour_cos']),
            'day_sin': float(row['day_sin']),
            'day_cos': float(row['day_cos']),
            'weather_code': int(row['weather_code']),
            'segment_multiplier': float(row['segment_multiplier']),
            
            # Network (if available)
            'network_': len(network_attrs) > 0,
            'junction': junction,
            'service': service,
            'ref': ref,
            'maxspeed': maxspeed,
            
            # Impact indicators
            'impact_indicators': impact_indicators,
            
            # Searchable fields for IR
            'location_tokens': f"node_{source_node} node_{target_node} {network_highway} {road_name} {ref}",
            'weather_tokens': f"{weather_condition} rain precipitation temperature {temperature:.0f} humidity {humidity:.0f} wind {wind_speed:.0f}",
            'traffic_tokens': f"{congestion_level} congestion vehicles {vehicle_count:.0f} capacity {road_capacity} rush_hour {is_rush_hour}",
            'time_tokens': f"hour_{hour_of_day} day_{day_of_week} {time_context.lower()} weekend {is_weekend}",
            'road_tokens': f"{network_highway} {lanes} {maxspeed} {junction} {service} {road_name} speed_limit {speed_limit}",
            'impact_tokens': f"{' '.join(impact_indicators).lower()}",
            'coordinate_tokens': f"source_{source_x:.6f}_{source_y:.6f} target_{target_x:.6f}_{target_y:.6f}"
        }
        
        return document
    
    def generate_corpus(self, traffic_data: pd.DataFrame, sample_size: Optional[int] = None) -> List[Dict[str, Any]]:
        """Generate complete IR document corpus from traffic data"""
        print("Generating Traffic Event Corpus from Traffic Data Features...")
        print(f"Available features: {list(traffic_data.columns)}")
        
        # Sample data if specified
        if sample_size and len(traffic_data) > sample_size:
            working_data = traffic_data.sample(n=sample_size, random_state=42)
            print(f"Using sample of {sample_size:,} records from {len(traffic_data):,} total")
        else:
            working_data = traffic_data
            print(f"Processing all {len(working_data):,} records")
        
        documents = []
        processed_count = 0
        network__count = 0
        
        for idx, row in working_data.iterrows():
            doc_id = f"traffic_event_{idx}"
            
            try:
                document = self.create_event_document(row, doc_id)
                
                # Track network enhancements
                if document.get('network_', False):
                    network__count += 1
                
                documents.append(document)
                processed_count += 1
                
                # Progress reporting
                if processed_count % 10000 == 0:
                    print(f"Processed {processed_count:,} documents...")
                    
            except Exception as e:
                print(f"Error processing row {idx}: {e}")
                continue
        
        print(f"Generated {len(documents):,} traffic event documents")
        if self.edge_attributes:
            print(f"Network- documents: {network__count:,} ({network__count/len(documents)*100:.1f}%)")
        else:
            print("Using traffic data features only")
        
        return documents


In [66]:
class TrafficTextPreprocessor:
    """Text preprocessing with better token extraction"""
    
    def __init__(self):
        self.stemmer = PorterStemmer()
        self.traffic_stop_words = self._get__stop_words()
        self.traffic_keywords = self._get_traffic_keywords()
        self.token_stats = {}
        
    def _get__stop_words(self) -> Set[str]:
        """Get refined stop words - keep more meaningful terms"""
        # Standard English stop words
        standard_stop = set(stopwords.words('english'))
        
        # Remove some terms that are meaningful for traffic IR
        meaningful_terms = {'weather', 'traffic', 'congestion', 'delays', 'conditions'}
        standard_stop = standard_stop - meaningful_terms
        
        # Traffic-specific stop words (only truly non-searchable ones)
        traffic_stop = {
            'event', 'road', 'from', 'to', 'with', 'degrees', 'km', 'meters',
            'vehicles', 'capacity', 'temperature', 'precipitation', 'mm', 'wind', 'speed', 'direction',
            'visibility', 'pressure', 'hpa', 'humidity', 'percent', 'lanes', 'limit', 'along',
            'expected', 'impact', 'severe', 'moderate', 'light', 'heavy',
            'delays', 'hazardous', 'driving', 'conditions', 'peak', 'during'
        }
        
        # Combine and convert to lowercase
        all_stop_words = standard_stop.union(traffic_stop)
        return {word.lower() for word in all_stop_words}
    
    def _get_traffic_keywords(self) -> Dict[str, List[str]]:
        """Get domain-specific keywords and their variants"""
        return {
            'congestion': ['congestion', 'congested', 'traffic jam', 'traffic backup', 'gridlock'],
            'weather': ['rain', 'heavy rain', 'storm', 'clear', 'sunny', 'cloudy', 'fog', 'mist'],
            'road_types': ['motorway', 'highway', 'primary', 'secondary', 'tertiary', 'residential', 'service', 'trunk'],
            'time_periods': ['morning', 'evening', 'night', 'daytime', 'rush', 'peak', 'off-peak'],
            'impacts': ['delays', 'slow', 'fast', 'blocked', 'closed', 'detour', 'accident'],
            'visibility': ['poor visibility', 'good visibility', 'limited visibility', 'clear visibility'],
            'wind': ['strong wind', 'light wind', 'calm', 'breezy', 'gusty']
        }
    
    def _clean_text(self, text: str) -> str:
        """ text cleaning that preserves meaningful terms"""
        if not text or pd.isna(text):
            return ""
        
        # Convert to lowercase
        text = text.lower()
        
        # Preserve meaningful compound terms before splitting
        text = re.sub(r'heavy rain', 'heavy_rain', text)
        text = re.sub(r'poor visibility', 'poor_visibility', text)
        text = re.sub(r'morning rush', 'morning_rush', text)
        text = re.sub(r'evening rush', 'evening_rush', text)
        text = re.sub(r'rush hour', 'rush_hour', text)
        text = re.sub(r'roundabout', 'roundabout', text)  # Keep as single term
        
        # Handle traffic-specific patterns
        text = re.sub(r'node_(\d+)', r'node_\1', text)                # Keep node_ prefix
        text = re.sub(r'(\d+\.?\d*)\s*km', r'\1_km', text)            # Distance units
        text = re.sub(r'(\d+)\s*meters', r'\1_meters', text)          # Length units
        text = re.sub(r'(\d+)\s*vehicles', r'\1_vehicles', text)      # Vehicle counts
        text = re.sub(r'(\d+)\.?\d*\s*degrees', r'\1_degrees', text)  # Temperature
        text = re.sub(r'(\d+)\.?\d*\s*mm', r'\1_mm', text)            # Precipitation
        text = re.sub(r'(\d+)\s*km/h', r'\1_kmh', text)               # Speed
        text = re.sub(r'(\d+)\s*hpa', r'\1_hpa', text)                # Pressure
        
        # Handle time patterns
        text = re.sub(r'(\d{4}-\d{2}-\d{2})', r'\1', text)  # Dates
        text = re.sub(r'(\d{2}:\d{2}:\d{2})', r'\1', text)  # Times
        
        # Remove punctuation except underscores and hyphens in compounds
        text = re.sub(r'[^\w\s\-_]', ' ', text)
        
        # Clean up extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def extract_meaningful_tokens(self, text: str) -> List[str]:
        """Extract meaningful tokens using  patterns"""
        if not text:
            return []
        
        # Split into tokens
        tokens = text.split()
        meaningful_tokens = []
        
        for token in tokens:
            # Skip stop words
            if token.lower() in self.traffic_stop_words:
                continue
            
            # Skip pure numbers but keep meaningful number-based tokens
            if token.isdigit():
                continue
            
            # Skip very short tokens (except meaningful ones)
            if len(token) == 1 and token not in {'x', 'y', 'a', 'b'}:
                continue
            
            # Skip very long tokens (likely errors)
            if len(token) > 25:
                continue
            
            # Keep tokens with reasonable length
            if 2 <= len(token) <= 25:
                meaningful_tokens.append(token)
        
        return meaningful_tokens
    
    def extract_congestion_indicators(self, text: str) -> List[str]:
        """Extract congestion-related terms"""
        congestion_terms = []
        
        # Direct congestion mentions
        if 'heavy congestion' in text:
            congestion_terms.append('heavy_congestion')
        elif 'moderate congestion' in text:
            congestion_terms.append('moderate_congestion')
        elif 'light traffic' in text:
            congestion_terms.append('light_traffic')
        elif 'free flow' in text:
            congestion_terms.append('free_flow')
        
        # Impact indicators
        if 'severe delays' in text:
            congestion_terms.append('severe_delays')
        if 'peak traffic' in text:
            congestion_terms.append('peak_traffic')
        if 'roundabout congestion' in text:
            congestion_terms.append('roundabout_congestion')
        
        return congestion_terms
    
    def extract_weather_indicators(self, text: str) -> List[str]:
        """Extract weather-related terms"""
        weather_terms = []
        
        # Weather conditions
        if 'heavy rain' in text:
            weather_terms.append('heavy_rain')
        elif 'rain' in text:
            weather_terms.append('rain')
        elif 'clear' in text:
            weather_terms.append('clear')
        elif 'cold' in text:
            weather_terms.append('cold')
        elif 'hot' in text:
            weather_terms.append('hot')
        elif 'poor visibility' in text:
            weather_terms.append('poor_visibility')
        
        # Weather impacts
        if 'hazardous driving' in text:
            weather_terms.append('hazardous_driving')
        if 'rain during rush' in text:
            weather_terms.append('rain_rush_hour')
        
        return weather_terms
    
    def extract_spatial_indicators(self, text: str) -> List[str]:
        """Extract spatial and location-related terms"""
        spatial_terms = []
        
        # Node references
        node_pattern = r'node_(\d+)'
        nodes = re.findall(node_pattern, text)
        for node in nodes[:5]:  # Limit to prevent too many
            spatial_terms.append(f'node_{node}')
        
        # Road types
        road_types = ['motorway', 'primary', 'secondary', 'tertiary', 'residential', 'service', 'trunk']
        for road_type in road_types:
            if road_type in text:
                spatial_terms.append(road_type)
        
        # Route references
        route_pattern = r'route\s+(\w+)'
        routes = re.findall(route_pattern, text)
        spatial_terms.extend(routes)
        
        return spatial_terms
    
    def extract_temporal_indicators(self, text: str) -> List[str]:
        """Extract time-related terms"""
        temporal_terms = []
        
        # Time periods
        if 'morning rush' in text:
            temporal_terms.append('morning_rush')
        elif 'evening rush' in text:
            temporal_terms.append('evening_rush')
        elif 'rush hour' in text:
            temporal_terms.append('rush_hour')
        elif 'night' in text:
            temporal_terms.append('night')
        elif 'daytime' in text:
            temporal_terms.append('daytime')
        
        # Day types
        if 'weekend' in text:
            temporal_terms.append('weekend')
        elif 'weekday' in text:
            temporal_terms.append('weekday')
        elif 'friday' in text:
            temporal_terms.append('friday')
        
        return temporal_terms
    
    def extract_numerical_features(self, text: str) -> List[str]:
        """Extract meaningful numerical features"""
        numerical_terms = []
        
        # Vehicle counts (meaningful ranges)
        vehicle_pattern = r'(\d+)\s*vehicles'
        vehicles = re.findall(vehicle_pattern, text)
        for vehicle_count in vehicles:
            count = int(vehicle_count)
            if count > 500:
                numerical_terms.append('high_vehicle_count')
            elif count > 200:
                numerical_terms.append('moderate_vehicle_count')
            elif count > 50:
                numerical_terms.append('low_vehicle_count')
        
        # Speed limits
        speed_pattern = r'speed limit\s+(\d+)'
        speeds = re.findall(speed_pattern, text)
        for speed in speeds:
            speed_int = int(speed)
            if speed_int >= 80:
                numerical_terms.append('high_speed_limit')
            elif speed_int >= 50:
                numerical_terms.append('medium_speed_limit')
            else:
                numerical_terms.append('low_speed_limit')
        
        # Temperature ranges
        temp_pattern = r'temperature:\s*(\d+\.?\d*)'
        temps = re.findall(temp_pattern, text)
        for temp in temps:
            temp_float = float(temp)
            if temp_float > 30:
                numerical_terms.append('high_temperature')
            elif temp_float < 5:
                numerical_terms.append('low_temperature')
            else:
                numerical_terms.append('moderate_temperature')
        
        return numerical_terms
    
    def _tokenize(self, text: str) -> Dict[str, List[str]]:
        """Tokenization with multiple token categories"""
        cleaned_text = self._clean_text(text)
        
        # Extract different types of tokens
        base_tokens = self.extract_meaningful_tokens(cleaned_text)
        congestion_tokens = self.extract_congestion_indicators(text)
        weather_tokens = self.extract_weather_indicators(text)
        spatial_tokens = self.extract_spatial_indicators(text)
        temporal_tokens = self.extract_temporal_indicators(text)
        numerical_tokens = self.extract_numerical_features(text)
        
        # Combine all tokens
        all_tokens = list(set(base_tokens + congestion_tokens + weather_tokens + 
                             spatial_tokens + temporal_tokens + numerical_tokens))
        
        return {
            'base_tokens': base_tokens,
            'congestion_tokens': congestion_tokens,
            'weather_tokens': weather_tokens,
            'spatial_tokens': spatial_tokens,
            'temporal_tokens': temporal_tokens,
            'numerical_tokens': numerical_tokens,
            'all_tokens': all_tokens
        }
    
    def preprocess_document_(self, document: Dict[str, Any]) -> Dict[str, Any]:
        """Document preprocessing"""
        text = document.get('text', '')
        
        #  tokenization
        token_result = self._tokenize(text)
        
        # Create searchable text from all tokens
        searchable_text = ' '.join(token_result['all_tokens'])
        
        # Update document with  tokens
        processed_doc = document.copy()
        processed_doc.update({
            'cleaned_text': self._clean_text(text),
            'tokens': token_result['base_tokens'],
            'congestion_tokens': token_result['congestion_tokens'],
            'weather_tokens': token_result['weather_tokens'],
            'spatial_tokens': token_result['spatial_tokens'],
            'temporal_tokens': token_result['temporal_tokens'],
            'numerical_tokens': token_result['numerical_tokens'],
            'all_tokens': token_result['all_tokens'],
            'token_count': len(token_result['all_tokens']),
            'unique_tokens': len(set(token_result['all_tokens'])),
            'searchable_text': searchable_text
        })
        
        return processed_doc
    
    def preprocess_corpus_(self, documents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """ corpus preprocessing"""
        print(f" preprocessing of {len(documents)} traffic event documents...")
        
        processed_documents = []
        token_counts = []
        category_counts = {
            'congestion': 0,
            'weather': 0,
            'spatial': 0,
            'temporal': 0,
            'numerical': 0
        }
        
        for i, doc in enumerate(documents):
            processed_doc = self.preprocess_document_(doc)
            processed_documents.append(processed_doc)
            token_counts.append(processed_doc['token_count'])
            
            # Track category usage
            if processed_doc['congestion_tokens']:
                category_counts['congestion'] += 1
            if processed_doc['weather_tokens']:
                category_counts['weather'] += 1
            if processed_doc['spatial_tokens']:
                category_counts['spatial'] += 1
            if processed_doc['temporal_tokens']:
                category_counts['temporal'] += 1
            if processed_doc['numerical_tokens']:
                category_counts['numerical'] += 1
            
            # Progress reporting
            if (i + 1) % 1000 == 0:
                print(f"Processed {i + 1:,} documents...")
        
        #  statistics
        self.token_stats = {
            'total_documents': len(processed_documents),
            'total_tokens': sum(token_counts),
            'avg_tokens_per_doc': np.mean(token_counts),
            'median_tokens_per_doc': np.median(token_counts),
            'min_tokens_per_doc': min(token_counts),
            'max_tokens_per_doc': max(token_counts),
            'unique_tokens_in_corpus': len(set(token for doc in processed_documents for token in doc['all_tokens'])),
            'category_coverage': category_counts
        }
        
        print(f"\n Preprocessing Statistics:")
        print(f"Total documents: {self.token_stats['total_documents']:,}")
        print(f"Total tokens: {self.token_stats['total_tokens']:,}")
        print(f"Average tokens per document: {self.token_stats['avg_tokens_per_doc']:.1f}")
        print(f"Unique tokens in corpus: {self.token_stats['unique_tokens_in_corpus']:,}")
        print(f"\nToken Category Coverage:")
        for category, count in category_counts.items():
            percentage = count / len(processed_documents) * 100
            print(f"  {category}: {count:,} ({percentage:.1f}%)")
        
        return processed_documents
    
    def analyze__vocabulary(self, documents: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Analyze  vocabulary"""
        print("\nAnalyzing  vocabulary...")
        
        # Collect tokens by category
        all_base_tokens = []
        all_congestion = []
        all_weather = []
        all_spatial = []
        all_temporal = []
        all_numerical = []
        
        for doc in documents:
            all_base_tokens.extend(doc.get('base_tokens', []))
            all_congestion.extend(doc.get('congestion_tokens', []))
            all_weather.extend(doc.get('weather_tokens', []))
            all_spatial.extend(doc.get('spatial_tokens', []))
            all_temporal.extend(doc.get('temporal_tokens', []))
            all_numerical.extend(doc.get('numerical_tokens', []))
        
        # Analyze each category
        vocab_analysis = {
            'base_tokens': Counter(all_base_tokens).most_common(15),
            'congestion_tokens': Counter(all_congestion).most_common(10),
            'weather_tokens': Counter(all_weather).most_common(10),
            'spatial_tokens': Counter(all_spatial).most_common(10),
            'temporal_tokens': Counter(all_temporal).most_common(10),
            'numerical_tokens': Counter(all_numerical).most_common(10)
        }
        
        print(f" Token Analysis:")
        for category, tokens in vocab_analysis.items():
            if tokens:
                print(f"\n{category.replace('_', ' ').title()}:")
                for token, count in tokens[:5]:
                    print(f"  {token}: {count}")
        
        return vocab_analysis

In [72]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, Optional
from datetime import datetime
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DataIngestionPipeline:
    """Complete pipeline for traffic data ingestion and IR corpus building"""
    
    def __init__(self, data_path: str, output_path: str, use_categorized_tokens: bool = True):
        self.data_path = Path(data_path)
        self.output_path = Path(output_path)
        self.output_path.mkdir(exist_ok=True)
        self.use_categorized_tokens = use_categorized_tokens
        
        # Initialize components
        self.traffic_loader = TrafficWeatherDataLoader(str(self.data_path))
        self.network_loader = KigaliNetworkLoader(str(self.data_path))
        self.doc_generator = TrafficEventDocumentGenerator(self.network_loader)
        
        # Choose preprocessor
        if use_categorized_tokens:
            self.preprocessor = TrafficTextPreprocessor()
            logger.info("Using preprocessing with categorized tokens")
        else:
            self.preprocessor = TrafficTextPreprocessor()
            logger.info("Using standard preprocessing")
        
        # Pipeline statistics
        self.pipeline_stats = {}
        self.start_time = None
        self.end_time = None
        
    def validate_data_paths(self) -> bool:
        """Validate that required data files exist"""
        logger.info("Validating data paths...")
        
        # Check for traffic data with multiple possible patterns
        traffic_patterns = [
            "**/traffic_weather_temporal.csv",
            "**/routiq/traffic_weather_temporal.csv",
            "**/routeiq/traffic_weather_temporal.csv"
        ]
        
        traffic_files = []
        for pattern in traffic_patterns:
            traffic_files = list(self.data_path.glob(pattern))
            if traffic_files:
                break
        
        if not traffic_files:
            logger.error("Traffic weather data file not found")
            # List what we did find
            all_csv_files = list(self.data_path.glob("**/*.csv"))
            if all_csv_files:
                logger.info(f"Found CSV files: {[str(f) for f in all_csv_files[:5]]}")
            return False
        
        # Check for network data with multiple possible patterns
        network_patterns = [
            "**/kigali_congested_network.pkl",
            "**/routeiq/kigali_congested_network.pkl",
            "**/routiq/kigali_congested_network.pkl"
        ]
        
        network_files = []
        for pattern in network_patterns:
            network_files = list(self.data_path.glob(pattern))
            if network_files:
                break
        
        if not network_files:
            logger.warning("Network graph file not found - will proceed without network enhancement")
        else:
            logger.info(f"Found network data: {network_files[0]}")
        
        logger.info(f"Found traffic data: {traffic_files[0]}")
        return True
    
    def load_and_analyze_data(self) -> Dict[str, Any]:
        """Step 1: Load and analyze raw data"""
        logger.info("Step 1: Loading and analyzing raw data...")
        
        # Load traffic data
        traffic_data = self.traffic_loader.load_traffic_weather_data()
        traffic_stats = self.traffic_loader.analyze_data_quality()
        
        # Load network data (optional)
        network_stats = {}
        try:
            network_graph = self.network_loader.load_network_graph()
            network_stats = self.network_loader.analyze_network_structure()
            logger.info("Network data loaded successfully")
        except Exception as e:
            logger.warning(f"Could not load network data: {e}")
            network_stats = {'error': str(e)}
        
        return {
            'traffic_data': traffic_data,
            'traffic_stats': traffic_stats,
            'network_stats': network_stats
        }
    
    def generate_documents(self, traffic_data: pd.DataFrame, sample_size: Optional[int] = None) -> Dict[str, Any]:
        """Step 2: Generate IR documents from traffic data"""
        logger.info("Step 2: Generating IR documents...")
        
        # Sample data if specified
        if sample_size and len(traffic_data) > sample_size:
            working_data = traffic_data.sample(n=sample_size, random_state=42)
            logger.info(f"Using sample of {sample_size:,} records from {len(traffic_data):,} total")
        else:
            working_data = traffic_data
            logger.info(f"Processing all {len(working_data):,} records")
        
        # Generate documents
        documents = self.doc_generator.generate_corpus(working_data)
        
        # Analyze document generation
        doc_stats = {
            'total_documents': len(documents),
            'input_records': len(working_data),
            'generation_rate': len(documents) / len(working_data) * 100,
            'sample_size_used': len(working_data)
        }
        
        # Document content analysis
        if documents:
            doc_lengths = [len(doc.get('text', '')) for doc in documents]
            doc_stats.update({
                'avg_doc_length': np.mean(doc_lengths),
                'median_doc_length': np.median(doc_lengths),
                'min_doc_length': min(doc_lengths),
                'max_doc_length': max(doc_lengths)
            })
        
        return {
            'documents': documents,
            'doc_stats': doc_stats
        }
    
    def preprocess_documents(self, documents: list) -> Dict[str, Any]:
        """Step 3: Preprocess documents for IR"""
        logger.info("Step 3: Preprocessing documents...")
        
        # Use the fixed preprocessing methods
        processed_docs = self.preprocessor.preprocess_corpus_(documents)
        vocab_stats = self.preprocessor.analyze__vocabulary(processed_docs)
        
        # Preprocessing statistics
        prep_stats = {
            'documents_processed': len(processed_docs),
            'documents_success_rate': len(processed_docs) / len(documents) * 100,
            'vocabulary_size': vocab_stats.get('unique_tokens_in_corpus', 0),
            'total_tokens': vocab_stats.get('total_tokens', 0),
            'avg_tokens_per_doc': vocab_stats.get('avg_tokens_per_doc', 0),
            'preprocessing_method': 'categorized' if self.use_categorized_tokens else 'standard'
        }
        
        return {
            'processed_documents': processed_docs,
            'vocab_stats': vocab_stats,
            'prep_stats': prep_stats
        }
    
    def save_corpus(self, processed_documents: list) -> Dict[str, str]:
        """Step 4: Save processed corpus to files"""
        logger.info("Step 4: Saving processed corpus...")
        
        output_files = {}
        
        # Save main corpus CSV
        corpus_file = self.output_path / "traffic_event_corpus.csv"
        
        # Prepare data for CSV
        csv_data = []
        for doc in processed_documents:
            csv_row = {
                'doc_id': doc['doc_id'],
                'title': doc['title'],
                'text': doc['text'],
                'timestamp': doc['timestamp'],
                'congestion_level': doc['congestion_level'],
                'weather_condition': doc['weather_condition'],
                'highway_type': doc['highway_type'],
                'vehicle_count': doc['vehicle_count'],
                'temperature': doc['temperature'],
                'precipitation': doc['precipitation'],
                'is_rush_hour': doc['is_rush_hour'],
                'is_weekend': doc['is_weekend'],
                'source_node': doc['source_node'],
                'target_node': doc['target_node'],
                'token_count': doc['token_count']
            }
            
            # Add categorized token fields
            csv_row.update({
                'all_tokens': ' '.join(doc.get('all_tokens', [])),
                'congestion_tokens': ' '.join(doc.get('congestion_tokens', [])),
                'weather_tokens': ' '.join(doc.get('weather_tokens', [])),
                'spatial_tokens': ' '.join(doc.get('spatial_tokens', [])),
                'temporal_tokens': ' '.join(doc.get('temporal_tokens', [])),
                'numerical_tokens': ' '.join(doc.get('numerical_tokens', []))
            })
            
            csv_data.append(csv_row)
        
        # Save to CSV
        df = pd.DataFrame(csv_data)
        df.to_csv(corpus_file, index=False)
        output_files['corpus_csv'] = str(corpus_file)
        logger.info(f"Saved {len(csv_data):,} documents to {corpus_file}")
        
        # Save vocabulary statistics
        vocab_file = self.output_path / "vocabulary_stats.json"
        vocab_stats = self.preprocessor.analyze__vocabulary(processed_documents)
        vocab_json = {}
        for category, counter in vocab_stats.items():
            if hasattr(counter, 'most_common'):
                vocab_json[category] = dict(counter.most_common(50))
            else:
                vocab_json[category] = counter
        
        with open(vocab_file, 'w') as f:
            json.dump(vocab_json, f, indent=2)
        output_files['vocab_stats'] = str(vocab_file)
        
        # Save sample documents for inspection
        sample_file = self.output_path / "sample_documents.json"
        sample_docs = []
        for i, doc in enumerate(processed_documents[:10]):  # First 10 documents
            sample_doc = {
                'doc_id': doc['doc_id'],
                'title': doc['title'],
                'text': doc['text'][:500] + '...' if len(doc['text']) > 500 else doc['text'],
                'tokens': doc.get('all_tokens', []),
                'congestion_level': doc['congestion_level'],
                'weather_condition': doc['weather_condition'],
                'vehicle_count': doc['vehicle_count']
            }
            sample_docs.append(sample_doc)
        
        with open(sample_file, 'w') as f:
            json.dump(sample_docs, f, indent=2)
        output_files['sample_docs'] = str(sample_file)
        
        return output_files
    
    def generate_pipeline_report(self, all_stats: Dict[str, Any], output_files: Dict[str, str]) -> Dict[str, Any]:
        """Generate comprehensive pipeline report"""
        logger.info("Generating pipeline report...")
        
        # Calculate processing time
        processing_time = self.end_time - self.start_time if self.end_time and self.start_time else None
        
        # Comprehensive summary
        summary = {
            'pipeline_info': {
                'start_time': self.start_time.isoformat() if self.start_time else None,
                'end_time': self.end_time.isoformat() if self.end_time else None,
                'processing_time_seconds': processing_time.total_seconds() if processing_time else None,
                'preprocessing_method': 'categorized' if self.use_categorized_tokens else 'standard'
            },
            'data_ingestion_stats': all_stats.get('data_stats', {}),
            'document_generation_stats': all_stats.get('doc_stats', {}),
            'preprocessing_stats': all_stats.get('prep_stats', {}),
            'vocabulary_stats': all_stats.get('vocab_stats', {}),
            'output_files': output_files,
            'quality_metrics': {
                'data_quality_score': self._calculate_data_quality_score(all_stats.get('data_stats', {})),
                'document_quality_score': self._calculate_document_quality_score(all_stats.get('doc_stats', {})),
                'vocabulary_quality_score': self._calculate_vocabulary_quality_score(all_stats.get('vocab_stats', {}))
            }
        }
        
        # Save summary report
        summary_file = self.output_path / "pipeline_summary.json"
        with open(summary_file, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        
        return summary
    
    def _calculate_data_quality_score(self, data_stats: Dict[str, Any]) -> float:
        """Calculate data quality score (0-100)"""
        if not data_stats:
            return 0.0
        
        score = 0.0
        
        # Missing values (lower is better)
        missing_ratio = data_stats.get('missing_values', 0) / data_stats.get('total_rows', 1)
        score += (1 - min(missing_ratio, 0.1)) * 30  # Max 30 points
        
        # Duplicate rows (lower is better)
        duplicate_ratio = data_stats.get('duplicate_rows', 0) / data_stats.get('total_rows', 1)
        score += (1 - min(duplicate_ratio, 0.05)) * 20  # Max 20 points
        
        # Time coverage (higher is better)
        time_hours = data_stats.get('date_range', {}).get('hours', 0)
        score += min(time_hours / 168, 1) * 25  # Max 25 points (1 week = 168 hours)
        
        # Data variety (higher is better)
        unique_sources = data_stats.get('network_stats', {}).get('unique_sources', 0)
        score += min(unique_sources / 1000, 1) * 25  # Max 25 points
        
        return min(score, 100.0)
    
    def _calculate_document_quality_score(self, doc_stats: Dict[str, Any]) -> float:
        """Calculate document quality score (0-100)"""
        if not doc_stats:
            return 0.0
        
        score = 0.0
        
        # Generation success rate (higher is better)
        success_rate = doc_stats.get('generation_rate', 0)
        score += success_rate * 40  # Max 40 points
        
        # Average document length (moderate is better)
        avg_length = doc_stats.get('avg_doc_length', 0)
        optimal_length = 300  # Target around 300 characters
        length_score = max(0, 1 - abs(avg_length - optimal_length) / optimal_length)
        score += length_score * 30  # Max 30 points
        
        # Document variety (higher is better)
        total_docs = doc_stats.get('total_documents', 0)
        score += min(total_docs / 10000, 1) * 30  # Max 30 points
        
        return min(score, 100.0)
    
    def _calculate_vocabulary_quality_score(self, vocab_stats: Dict[str, Any]) -> float:
        """Calculate vocabulary quality score (0-100)"""
        if not vocab_stats:
            return 0.0
        
        score = 0.0
        
        # Vocabulary size (moderate is better)
        vocab_size = vocab_stats.get('unique_tokens_in_corpus', 0)
        size_score = min(vocab_size / 2000, 1)  # Target around 2000 unique tokens
        score += size_score * 40  # Max 40 points
        
        # Average tokens per document (moderate is better)
        avg_tokens = vocab_stats.get('avg_tokens_per_doc', 0)
        optimal_tokens = 20  # Target around 20 tokens per doc
        tokens_score = max(0, 1 - abs(avg_tokens - optimal_tokens) / optimal_tokens)
        score += tokens_score * 30  # Max 30 points
        
        # Token diversity (higher is better)
        total_tokens = vocab_stats.get('total_tokens', 0)
        if total_tokens > 0:
            diversity = vocab_size / total_tokens
            score += min(diversity * 100, 1) * 30  # Max 30 points
        
        return min(score, 100.0)
    
    def run_complete_pipeline(self, sample_size: Optional[int] = 10000) -> Dict[str, Any]:
        """Run complete data ingestion pipeline"""
        self.start_time = datetime.now()
        logger.info("Starting Complete Data Ingestion Pipeline")
        logger.info("=" * 60)
        
        try:
            # Step 0: Validate paths
            if not self.validate_data_paths():
                raise ValueError("Data validation failed")
            
            # Step 1: Load and analyze data
            data_results = self.load_and_analyze_data()
            
            # Step 2: Generate documents
            doc_results = self.generate_documents(data_results['traffic_data'], sample_size)
            
            # Step 3: Preprocess documents
            prep_results = self.preprocess_documents(doc_results['documents'])
            
            # Step 4: Save corpus
            output_files = self.save_corpus(prep_results['processed_documents'])
            
            # Step 5: Generate report
            all_stats = {
                'data_stats': data_results['traffic_stats'],
                'doc_stats': doc_results['doc_stats'],
                'prep_stats': prep_results['prep_stats'],
                'vocab_stats': prep_results['vocab_stats']
            }
            
            self.end_time = datetime.now()
            summary = self.generate_pipeline_report(all_stats, output_files)
            
            # Log final results
            logger.info("\nData Ingestion Pipeline Complete!")
            logger.info(f"Processing time: {(self.end_time - self.start_time).total_seconds():.2f} seconds")
            logger.info(f"Documents generated: {doc_results['doc_stats']['total_documents']:,}")
            logger.info(f"Vocabulary size: {prep_results['prep_stats']['vocabulary_size']:,}")
            logger.info(f"Average tokens per document: {prep_results['prep_stats']['avg_tokens_per_doc']:.1f}")
            logger.info(f"Output files: {list(output_files.values())}")
            
            return summary
            
        except Exception as e:
            logger.error(f"Pipeline failed: {e}")
            self.end_time = datetime.now()
            raise

# Usage Examples
if __name__ == "__main__":
    # Use the correct data path from your previous sessions
    DATA_PATH = "/kaggle/input/datasets/lh7ng0cifjog1/routiq"
    
    # Example 1: Standard pipeline with sample data
    print("Example 1: Standard Pipeline")
    pipeline_standard = DataIngestionPipeline(
        data_path=DATA_PATH,
        output_path="./processed_data_standard",
        use_categorized_tokens=False
    )
    
    summary_standard = pipeline_standard.run_complete_pipeline(sample_size=5000)
    
    # Example 2: Pipeline with categorized tokens
    print("\nExample 2: Pipeline with Categorized Tokens")
    pipeline_categorized = DataIngestionPipeline(
        data_path=DATA_PATH,
        output_path="./processed_data_categorized",
        use_categorized_tokens=True
    )
    
    summary_categorized = pipeline_categorized.run_complete_pipeline(sample_size=10000)
    
    print("\nPipeline Summaries:")
    print(f"Standard: {summary_standard['document_generation_stats']['total_documents']:,} docs")
    print(f"Categorized: {summary_categorized['document_generation_stats']['total_documents']:,} docs")

2026-04-24 09:36:14,866 - INFO - Using standard preprocessing
2026-04-24 09:36:14,867 - INFO - Starting Complete Data Ingestion Pipeline
2026-04-24 09:36:14,867 - INFO - ============================================================
2026-04-24 09:36:14,868 - INFO - Validating data paths...
2026-04-24 09:36:14,881 - WARNING - Network graph file not found - will proceed without network enhancement
2026-04-24 09:36:14,882 - INFO - Found traffic data: /kaggle/input/datasets/lh7ng0cifjog1/routiq/traffic_weather_temporal.csv
2026-04-24 09:36:14,882 - INFO - Step 1: Loading and analyzing raw data...


Example 1: Standard Pipeline
Loading Traffic-Weather Temporal Data...
Found data at: /kaggle/input/datasets/lh7ng0cifjog1/routiq/traffic_weather_temporal.csv
Loaded 544,320 traffic records
Time range: 2026-02-10 00:00:00 to 2026-02-16 23:00:00
Network: 1023 source nodes, 1023 target nodes

Data Quality Analysis...


2026-04-24 09:36:20,145 - INFO - Network data loaded successfully
2026-04-24 09:36:20,146 - INFO - Step 2: Generating IR documents...
2026-04-24 09:36:20,164 - INFO - Using sample of 5,000 records from 544,320 total


Dataset Size: 544,320 rows × 41 columns
Memory Usage: 166.9 MB
Missing Values: 0
Duplicate Rows: 0
Time Coverage: 167.0 hours
Rainy Records: 353,160 (64.9%)
Rush Hour Records: 48,600
Loading Kigali Road Network Graph...
Found data at: /kaggle/input/datasets/lh7ng0cifjog1/routeiq/kigali_congested_network.pkl
Loaded road network with 1,026 nodes and 2,544 edges
Graph Type: MultiDiGraph
Connected Components: 10

Network Structure Analysis...
Analyzing 2544 edges with 12 attributes
Network: 1,026 nodes, 2,544 edges
Density: 0.0024
Avg Degree: 4.96
Connected Components: 10
Generating Traffic Event Corpus from Traffic Data Features...
Available features: ['source_node', 'target_node', 'timestamp', 'day_of_week', 'hour_of_day', 'highway_type', 'road_capacity', 'road_length_meters', 'speed_limit_kmh', 'lanes', 'vehicle_counts', 'is_weekend', 'is_rush_hour', 'temperature', 'humidity', 'pressure', 'wind_speed', 'wind_direction', 'visibility', 'precipitation', 'cloud_cover', 'day_of_week_num', 'h

2026-04-24 09:36:21,039 - INFO - Step 3: Preprocessing documents...


...

Processed 10,000 documents...
Generated 10,000 traffic event documents
Using traffic data features only
 preprocessing of 10000 traffic event documents...
Processed 1,000 documents...
Processed 2,000 documents...
Processed 3,000 documents...
Processed 4,000 documents...
Processed 5,000 documents...
Processed 6,000 documents...
Processed 7,000 documents...
Processed 8,000 documents...
Processed 9,000 documents...


2026-04-24 09:36:36,978 - INFO - Step 4: Saving processed corpus...


Processed 10,000 documents...

 Preprocessing Statistics:
Total documents: 10,000
Total tokens: 229,678
Average tokens per document: 23.0
Unique tokens in corpus: 1,837

Token Category Coverage:
  congestion: 0 (0.0%)
  weather: 1,902 (19.0%)
  spatial: 9,603 (96.0%)
  temporal: 779 (7.8%)
  numerical: 10,000 (100.0%)

Analyzing  vocabulary...
 Token Analysis:

Weather Tokens:
  rain: 1902

Spatial Tokens:
  residential: 5090
  service: 2349
  secondary: 1507
  trunk: 288
  tertiary: 266

Temporal Tokens:
  rush_hour: 779

Numerical Tokens:
  medium_speed_limit: 9291
  low_vehicle_count: 5904
  moderate_vehicle_count: 2581
  low_speed_limit: 709
  high_vehicle_count: 577


2026-04-24 09:36:37,347 - INFO - Saved 10,000 documents to processed_data_categorized/traffic_event_corpus.csv
2026-04-24 09:36:37,365 - INFO - Generating pipeline report...
2026-04-24 09:36:37,366 - INFO - 
Data Ingestion Pipeline Complete!
2026-04-24 09:36:37,367 - INFO - Processing time: 13.13 seconds
2026-04-24 09:36:37,368 - INFO - Documents generated: 10,000
2026-04-24 09:36:37,368 - INFO - Vocabulary size: 0
2026-04-24 09:36:37,369 - INFO - Average tokens per document: 0.0
2026-04-24 09:36:37,370 - INFO - Output files: ['processed_data_categorized/traffic_event_corpus.csv', 'processed_data_categorized/vocabulary_stats.json', 'processed_data_categorized/sample_documents.json']



Analyzing  vocabulary...
 Token Analysis:

Weather Tokens:
  rain: 1902

Spatial Tokens:
  residential: 5090
  service: 2349
  secondary: 1507
  trunk: 288
  tertiary: 266

Temporal Tokens:
  rush_hour: 779

Numerical Tokens:
  medium_speed_limit: 9291
  low_vehicle_count: 5904
  moderate_vehicle_count: 2581
  low_speed_limit: 709
  high_vehicle_count: 577

Pipeline Summaries:
Standard: 5,000 docs
Categorized: 10,000 docs
